# Summary_Day4.ipynb
## AI 기초: 신경망, 역전파(Backprop), 활성화 함수

이번 4차시는 실제 머신러닝 학습 루프를 처음부터 끝까지 구현합니다.

핵심 흐름:

```text
Forward → Loss → Backward → Update
```

구성:

1. AI / ML / DL 관계
2. 신장 → 체중 예측 선형회귀
3. 손실 함수 MSE
4. 역전파와 gradient
5. 직접 파라미터 업데이트
6. Optimizer 사용
7. Momentum 비교
8. 활성화 함수
9. Bias-Variance
10. Train / Validation / Test 분할

## 1. 라이브러리 준비

NumPy, Matplotlib, PyTorch를 불러옵니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import math

%matplotlib inline

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

np.set_printoptions(suppress=True, precision=4)

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)

## 2. AI / ML / DL 관계

포함 관계는 다음과 같습니다.

```text
AI ⊃ ML ⊃ DL
```

- AI: 사람처럼 판단하거나 행동하는 넓은 개념
- ML: 데이터를 보고 규칙을 학습하는 AI
- DL: 신경망을 여러 층 쌓아 복잡한 패턴을 학습하는 ML

In [ ]:
concepts = {
    "AI": "Artificial Intelligence, 인공지능",
    "ML": "Machine Learning, 머신러닝",
    "DL": "Deep Learning, 딥러닝"
}

for key, value in concepts.items():
    print(f"{key}: {value}")

## 3. 학습 방식 정리

이번 실습은 입력과 정답이 함께 있는 **지도학습**입니다.

```text
입력 X = 신장
정답 Y = 체중
```

In [ ]:
learning_types = [
    ("지도학습", "입력과 정답이 함께 주어진 데이터로 학습"),
    ("비지도학습", "정답 없이 데이터의 숨겨진 패턴을 찾음"),
    ("자기지도학습", "데이터 자체에서 문제와 정답을 만들어 학습"),
    ("강화학습", "행동 후 보상을 받으며 더 좋은 행동을 학습")
]

for name, desc in learning_types:
    print(f"{name}: {desc}")

## 4. 선형 모델

가장 기본 모델은 다음과 같습니다.

```text
Yp = W * X + B
```

- `X`: 입력
- `Yp`: 예측값
- `W`: 가중치
- `B`: 편향

In [ ]:
def linear_model(X, W, B):
    return W * X + B

sample_X = torch.tensor([1.0, 2.0, 3.0])
sample_W = torch.tensor(2.0)
sample_B = torch.tensor(1.0)

print(linear_model(sample_X, sample_W, sample_B))

## 5. 데이터 준비

신장과 체중 데이터를 2차원 배열로 만듭니다.

In [ ]:
sampleData1 = np.array([
    [166, 58.7],
    [176.0, 75.7],
    [171.0, 62.1],
    [173.0, 70.4],
    [169.0, 60.1]
])

print(sampleData1)
print("shape:", sampleData1.shape)

## 6. 입력과 정답 분리

- `x`: 신장
- `y`: 체중

NumPy 슬라이싱으로 열을 분리합니다.

In [ ]:
x = sampleData1[:, 0]
y = sampleData1[:, 1]

print("x:", x)
print("y:", y)

## 7. 원본 데이터 산점도

신장과 체중의 관계를 점으로 확인합니다.

In [ ]:
plt.scatter(x, y, c="k", s=50)
plt.xlabel("height(cm)")
plt.ylabel("weight(kg)")
plt.title("Height and Weight")
plt.show()

## 8. 데이터 전처리: 평균 빼기

경사하강법은 값이 너무 크면 불안정할 수 있습니다.

그래서 평균을 빼서 데이터 중심을 0으로 옮깁니다.

In [ ]:
X_np = x - x.mean()
Y_np = y - y.mean()

print("X:", X_np)
print("Y:", Y_np)
print("X mean:", X_np.mean())
print("Y mean:", Y_np.mean())

## 9. 전처리 후 산점도

데이터의 중심은 0 근처로 이동하지만 관계는 유지됩니다.

In [ ]:
plt.scatter(X_np, Y_np, c="k", s=50)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Centered Data")
plt.show()

## 10. Tensor 변환

PyTorch 학습을 위해 NumPy 배열을 Tensor로 바꿉니다.

In [ ]:
X = torch.tensor(X_np).float()
Y = torch.tensor(Y_np).float()

print(X)
print(Y)
print(X.dtype, Y.dtype)

## 11. 학습 파라미터 W, B 생성

`requires_grad=True`를 설정해야 PyTorch가 gradient를 계산합니다.

In [ ]:
W = torch.tensor(1.0, requires_grad=True).float()
B = torch.tensor(1.0, requires_grad=True).float()

print("W:", W)
print("B:", B)
print("W requires_grad:", W.requires_grad)
print("B requires_grad:", B.requires_grad)

## 12. 예측 함수

현재 W와 B로 예측값을 계산합니다.

In [ ]:
def pred(X):
    return W * X + B

Yp = pred(X)

print(Yp)

## 13. MSE 손실 함수

MSE는 예측값과 정답의 차이를 제곱한 뒤 평균낸 값입니다.

In [ ]:
def mse(Yp, Y):
    loss = ((Yp - Y) ** 2).mean()
    return loss

loss = mse(Yp, Y)

print(loss)

## 14. 역전파로 gradient 계산

`loss.backward()`를 호출하면 W와 B의 gradient가 계산됩니다.

In [ ]:
loss.backward()

print("W.grad:", W.grad)
print("B.grad:", B.grad)

## 15. 잘못된 업데이트 방식 확인

`requires_grad=True`인 Tensor를 그냥 `-=`로 직접 수정하면 에러가 발생합니다.

In [ ]:
lr = 0.001

try:
    W -= lr * W.grad
    B -= lr * B.grad
except RuntimeError as e:
    print("에러 발생:")
    print(e)

## 16. 올바른 업데이트 방식

파라미터 수정은 `torch.no_grad()` 안에서 수행합니다.

수정 후에는 gradient를 초기화합니다.

In [ ]:
with torch.no_grad():
    W -= lr * W.grad
    B -= lr * B.grad

W.grad.zero_()
B.grad.zero_()

print("W:", W)
print("B:", B)
print("W.grad:", W.grad)
print("B.grad:", B.grad)

## 17. 반복 학습 준비

학습 루프를 돌리기 위해 W, B, epoch, learning rate, history를 초기화합니다.

In [ ]:
W = torch.tensor(1.0, requires_grad=True).float()
B = torch.tensor(1.0, requires_grad=True).float()

num_epochs = 500
lr = 0.001
history = np.zeros((0, 2))

print("초기 W:", W.item())
print("초기 B:", B.item())

## 18. 직접 구현한 학습 루프

학습 흐름:

```text
예측 → 손실 → 역전파 → 업데이트 → 초기화
```

In [ ]:
for epoch in range(num_epochs):
    Yp = pred(X)
    loss = mse(Yp, Y)
    loss.backward()

    with torch.no_grad():
        W -= lr * W.grad
        B -= lr * B.grad

    W.grad.zero_()
    B.grad.zero_()

    if epoch % 10 == 0:
        history = np.vstack((history, np.array([epoch, loss.item()])))

print("최종 W:", W.item())
print("최종 B:", B.item())
print("초기 loss:", history[0, 1])
print("최종 loss:", history[-1, 1])

## 19. 학습 곡선

Loss가 반복에 따라 줄어드는지 확인합니다.

In [ ]:
plt.plot(history[:, 0], history[:, 1])
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Training Loss")
plt.show()

## 20. 학습된 직선 확인

전처리된 데이터 기준으로 학습된 직선을 그립니다.

In [ ]:
X_range = torch.tensor([X.min(), X.max()]).float()
Y_range = pred(X_range)

plt.scatter(X, Y, c="k", s=50)
plt.plot(X_range.data, Y_range.data, linewidth=2)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Fitted Line after Centering")
plt.show()

## 21. 원본 스케일로 직선 확인

평균을 다시 더해 원래 신장/체중 그래프 위에 직선을 그립니다.

In [ ]:
x_range = X_range + x.mean()
yp_range = Y_range + y.mean()

plt.scatter(x, y, c="k", s=50)
plt.plot(x_range, yp_range.data, linewidth=2)
plt.xlabel("height(cm)")
plt.ylabel("weight(kg)")
plt.title("Fitted Line in Original Scale")
plt.show()

## 22. Optimizer 사용

직접 `W -= lr * W.grad`를 쓰지 않고 Optimizer가 업데이트를 대신하게 합니다.

In [ ]:
W = torch.tensor(1.0, requires_grad=True).float()
B = torch.tensor(1.0, requires_grad=True).float()

optimizer = optim.SGD([W, B], lr=lr)
history_optimizer = np.zeros((0, 2))

print(optimizer)

## 23. Optimizer 학습 루프

핵심은 `optimizer.step()`과 `optimizer.zero_grad()`입니다.

In [ ]:
for epoch in range(num_epochs):
    Yp = pred(X)
    loss = mse(Yp, Y)
    loss.backward()

    optimizer.step()
    optimizer.zero_grad()

    if epoch % 10 == 0:
        history_optimizer = np.vstack((history_optimizer, np.array([epoch, loss.item()])))

print("최종 W:", W.item())
print("최종 B:", B.item())
print("초기 loss:", history_optimizer[0, 1])
print("최종 loss:", history_optimizer[-1, 1])

## 24. Optimizer 학습 곡선

Optimizer를 사용해도 Loss가 감소하는지 확인합니다.

In [ ]:
plt.plot(history_optimizer[:, 0], history_optimizer[:, 1])
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Training Loss with Optimizer")
plt.show()

## 25. Momentum 적용

Momentum은 이전 이동 방향의 관성을 반영해 학습을 가속할 수 있습니다.

In [ ]:
history_default = history_optimizer.copy()

W = torch.tensor(1.0, requires_grad=True).float()
B = torch.tensor(1.0, requires_grad=True).float()

optimizer = optim.SGD([W, B], lr=lr, momentum=0.9)
history_momentum = np.zeros((0, 2))

for epoch in range(num_epochs):
    Yp = pred(X)
    loss = mse(Yp, Y)
    loss.backward()

    optimizer.step()
    optimizer.zero_grad()

    if epoch % 10 == 0:
        history_momentum = np.vstack((history_momentum, np.array([epoch, loss.item()])))

print("Momentum 최종 loss:", history_momentum[-1, 1])

## 26. SGD vs Momentum 비교

두 학습 곡선을 비교합니다.

In [ ]:
plt.plot(history_default[:, 0], history_default[:, 1], label="SGD")
plt.plot(history_momentum[:, 0], history_momentum[:, 1], label="SGD + momentum")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.title("SGD vs Momentum")
plt.show()

## 27. 활성화 함수

활성화 함수는 신경망에 비선형성을 추가합니다.

활성화 함수가 없으면 깊은 신경망도 결국 하나의 선형 모델입니다.

In [ ]:
x_act = torch.linspace(-3, 3, 100)

sigmoid = torch.sigmoid(x_act)
tanh = torch.tanh(x_act)
relu = torch.relu(x_act)

plt.plot(x_act, sigmoid, label="Sigmoid")
plt.plot(x_act, tanh, label="Tanh")
plt.plot(x_act, relu, label="ReLU")
plt.legend()
plt.xlabel("x")
plt.ylabel("activation(x)")
plt.title("Activation Functions")
plt.show()

## 28. MLP 구조

신경망은 보통 다음 구조를 반복합니다.

```text
Linear → Activation → Linear → Activation
```

In [ ]:
model_example = nn.Sequential(
    nn.Linear(1, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

print(model_example)

## 29. Bias-Variance 개념

- 과소적합: 모델이 너무 단순해서 패턴을 못 배움
- 과적합: 모델이 훈련 데이터를 너무 외움

In [ ]:
bias_variance = {
    "High Bias": "모델이 너무 단순해서 패턴을 제대로 못 배움",
    "High Variance": "모델이 너무 복잡해서 훈련 데이터에 민감함",
    "Underfitting": "Train loss와 Validation loss가 모두 높음",
    "Overfitting": "Train loss는 낮지만 Validation loss가 높음"
}

for key, value in bias_variance.items():
    print(f"{key}: {value}")

## 30. 사인파 데이터 생성

Bias-Variance 실습을 위해 노이즈가 있는 사인파 데이터를 만듭니다.

In [ ]:
torch.manual_seed(0)

N = 600
x_sin = torch.linspace(-3 * math.pi, 3 * math.pi, N).unsqueeze(1)
y_sin = torch.sin(x_sin) + 0.2 * torch.randn_like(x_sin)

print("x_sin shape:", x_sin.shape)
print("y_sin shape:", y_sin.shape)

## 31. 사인파 데이터 시각화

In [ ]:
plt.scatter(x_sin, y_sin, s=8)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Sine Data with Noise")
plt.show()

## 32. Train / Validation / Test 분할

- Train: 학습
- Validation: 튜닝
- Test: 최종 평가

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    x_sin.numpy(), y_sin.numpy(), test_size=0.4, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

## 33. 작은 MLP와 큰 MLP 만들기

In [ ]:
def make_mlp(hidden):
    return nn.Sequential(
        nn.Linear(1, hidden),
        nn.ReLU(),
        nn.Linear(hidden, hidden),
        nn.ReLU(),
        nn.Linear(hidden, 1)
    )

small = make_mlp(hidden=8)
big = make_mlp(hidden=128)

print("Small model:")
print(small)

print("\nBig model:")
print(big)

## 34. 학습 함수 만들기

Train loss와 Validation loss를 기록합니다.

In [ ]:
def train(model, Xtr, ytr, Xva, yva, epochs=400, lr=1e-3):
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    tr_hist, va_hist = [], []

    for ep in range(epochs):
        model.train()
        opt.zero_grad()

        pred_value = model(Xtr)
        loss = loss_fn(pred_value, ytr)
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(Xva), yva).item()

        tr_hist.append(loss.item())
        va_hist.append(val_loss)

    return tr_hist, va_hist

## 35. 작은 모델과 큰 모델 학습

In [ ]:
tr_s, va_s = train(small, X_train, y_train, X_val, y_val, epochs=400)
tr_b, va_b = train(big, X_train, y_train, X_val, y_val, epochs=400)

print("small train:", tr_s[-1], "small val:", va_s[-1])
print("big train:", tr_b[-1], "big val:", va_b[-1])

## 36. Bias-Variance 학습 곡선 비교

In [ ]:
plt.plot(tr_s, label="small-train")
plt.plot(va_s, label="small-val")
plt.plot(tr_b, label="big-train")
plt.plot(va_b, label="big-val")

plt.legend()
plt.xlabel("epoch")
plt.ylabel("MSE Loss")
plt.title("Bias-Variance")
plt.show()

## 37. Test MSE 계산

Test 데이터는 최종 평가에만 사용합니다.

In [ ]:
def test_mse(model, X, y):
    model.eval()
    with torch.no_grad():
        return nn.MSELoss()(model(X), y).item()

print("Small Test MSE:", test_mse(small, X_test, y_test))
print("Big Test MSE:", test_mse(big, X_test, y_test))

## 38. 주요 함수 / 변수 / 약어 정리

| 이름 | 의미 | 설명 |
|---|---|---|
| `X` | Input | 입력값 |
| `Y` | Target | 정답값 |
| `Yp` | Prediction | 예측값 |
| `W` | Weight | 가중치 |
| `B` | Bias | 편향 |
| `loss` | Loss | 오차 |
| `MSE` | Mean Squared Error | 평균 제곱 오차 |
| `lr` | Learning Rate | 학습률 |
| `grad` | Gradient | 기울기 |
| `epoch` | Epoch | 반복 학습 단위 |
| `optimizer` | Optimizer | 파라미터 수정 도구 |
| `SGD` | Stochastic Gradient Descent | 확률적 경사하강법 |
| `MLP` | Multi-Layer Perceptron | 다층 퍼셉트론 |
| `ReLU` | Rectified Linear Unit | 대표 활성화 함수 |

## 39. 시험용 요약

```text
Forward → Loss → Backward → Update
```

- 선형 모델 공식은 `Yp = W * X + B`입니다.
- MSE는 예측값과 정답의 차이를 제곱한 뒤 평균낸 손실 함수입니다.
- `loss.backward()`는 역전파로 gradient를 계산합니다.
- `.grad`에는 계산된 gradient가 저장됩니다.
- 직접 파라미터를 수정할 때는 `torch.no_grad()`를 사용합니다.
- gradient는 누적되므로 매 반복마다 초기화해야 합니다.
- `optimizer.step()`은 파라미터 수정입니다.
- `optimizer.zero_grad()`는 gradient 초기화입니다.
- 활성화 함수는 신경망에 비선형성을 추가합니다.
- 활성화 함수가 없으면 깊은 신경망도 하나의 선형 모델과 같습니다.
- 과소적합은 모델이 너무 단순한 상태입니다.
- 과적합은 모델이 훈련 데이터를 너무 외운 상태입니다.
- Train은 학습, Validation은 튜닝, Test는 최종 평가에 사용합니다.
- Test 데이터가 학습 과정에 섞이면 데이터 누수입니다.